In [0]:
%run /tcga_cancer_batch_databricks/00_configuracion

Configuración cargada correctamente.
Ruta base: /Volumes/workspace/default/tcga_cancer_ml
Número de clases oficiales: 18


Estructura creada correctamente:
/Volumes/workspace/default/tcga_cancer_ml
/Volumes/workspace/default/tcga_cancer_ml/raw
/Volumes/workspace/default/tcga_cancer_ml/trusted
/Volumes/workspace/default/tcga_cancer_ml/refined
/Volumes/workspace/default/tcga_cancer_ml/models
/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq
/Volumes/workspace/default/tcga_cancer_ml/raw/metadata
/Volumes/workspace/default/tcga_cancer_ml/trusted/rnaseq_long
/Volumes/workspace/default/tcga_cancer_ml/trusted/rnaseq_matrix
/Volumes/workspace/default/tcga_cancer_ml/refined/eda_outputs
/Volumes/workspace/default/tcga_cancer_ml/refined/model_metrics
/Volumes/workspace/default/tcga_cancer_ml/refined/predictions
/Volumes/workspace/default/tcga_cancer_ml/refined/visualizations


In [0]:
import os
import json
import time
import requests
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

FILES_ENDPT = "https://api.gdc.cancer.gov/files"
DATA_ENDPT = "https://api.gdc.cancer.gov/data"

print("Ruta raw RNA-Seq:", RAW_RNASEQ_PATH)
print("Ruta raw metadata:", RAW_METADATA_PATH)

Ruta raw RNA-Seq: /Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq
Ruta raw metadata: /Volumes/workspace/default/tcga_cancer_ml/raw/metadata


In [0]:
# Consultar archivos RNA-Seq STAR Counts en GDC

filters = {
    "op": "and",
    "content": [
        {
            "op": "in",
            "content": {
                "field": "cases.project.project_id",
                "value": PROYECTOS_PRINCIPALES
            }
        },
        {
            "op": "in",
            "content": {
                "field": "data_category",
                "value": ["Transcriptome Profiling"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "data_type",
                "value": ["Gene Expression Quantification"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "analysis.workflow_type",
                "value": ["STAR - Counts"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "access",
                "value": ["open"]
            }
        }
    ]
}

fields = [
    "file_id",
    "file_name",
    "file_size",
    "cases.case_id",
    "cases.submitter_id",
    "cases.project.project_id",
    "cases.samples.sample_id",
    "cases.samples.submitter_id",
    "cases.samples.sample_type"
]

params = {
    "filters": json.dumps(filters),
    "fields": ",".join(fields),
    "format": "JSON",
    "size": "20000"
}

response = requests.get(FILES_ENDPT, params=params)
response.raise_for_status()

data = response.json()
archivos = data["data"]["hits"]

print("Archivos encontrados en GDC:", len(archivos))

Archivos encontrados en GDC: 9580


In [0]:
# Crear tabla de metadatos desde respuesta de GDC

registros = []

for archivo in archivos:
    file_id = archivo.get("file_id")
    file_name = archivo.get("file_name")
    file_size = archivo.get("file_size")

    cases = archivo.get("cases", [])

    if len(cases) == 0:
        continue

    caso = cases[0]

    case_id = caso.get("case_id")
    case_submitter_id = caso.get("submitter_id")

    project = caso.get("project", {})
    project_id = project.get("project_id")

    cancer_type = mapa_cancer.get(project_id, "DESCONOCIDO")
    cancer_name = mapa_nombre_cancer.get(project_id, "DESCONOCIDO")

    samples = caso.get("samples", [])

    if len(samples) > 0:
        sample_id = samples[0].get("sample_id")
        sample_submitter_id = samples[0].get("submitter_id")
        sample_type = samples[0].get("sample_type")
    else:
        sample_id = None
        sample_submitter_id = None
        sample_type = None

    registros.append({
        "file_id": file_id,
        "file_name": file_name,
        "file_size": file_size,
        "case_id": case_id,
        "case_submitter_id": case_submitter_id,
        "sample_id": sample_id,
        "sample_submitter_id": sample_submitter_id,
        "sample_type": sample_type,
        "project_id": project_id,
        "cancer_type": cancer_type,
        "cancer_name": cancer_name
    })

df_metadatos = pd.DataFrame(registros)

print("Dimensiones de metadatos:")
print(df_metadatos.shape)

display(df_metadatos.head())

print("Conteo por tipo de cáncer:")
display(
    df_metadatos["cancer_type"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "cancer_type", "cancer_type": "n_archivos"})
)

Dimensiones de metadatos:
(9580, 11)


file_id,file_name,file_size,case_id,case_submitter_id,sample_id,sample_submitter_id,sample_type,project_id,cancer_type,cancer_name
744a6d3d-b666-49aa-8d26-47f34e3d1eb5,94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.augmented_star_gene_counts.tsv,4231952,878f975b-94fd-4d69-b7e7-1ed3ac2ee438,TCGA-BH-A18H,9321161b-d568-4d38-aed3-956786a06329,TCGA-BH-A18H-01A,Primary Tumor,TCGA-BRCA,BRCA,Breast invasive carcinoma
4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5,df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.augmented_star_gene_counts.tsv,4238832,e4fc0909-f284-4471-866d-d8967b6adcbc,TCGA-E2-A14P,58bfe278-a80f-4722-b286-4d966214d244,TCGA-E2-A14P-01A,Primary Tumor,TCGA-BRCA,BRCA,Breast invasive carcinoma
1ace2a0c-773d-45b5-8fd6-968c88731bbb,c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.augmented_star_gene_counts.tsv,4239626,e5aae05a-478e-4a55-a27c-12b2b4be302a,TCGA-AN-A04A,1d8f72b7-8596-40e8-afdf-244dd897758c,TCGA-AN-A04A-01A,Primary Tumor,TCGA-BRCA,BRCA,Breast invasive carcinoma
7db46624-6d07-4f49-b0ff-18bb41375de9,b1f7af2a-a39b-44c7-a077-e1df6b18268a.rna_seq.augmented_star_gene_counts.tsv,4236752,7ef7c87e-6f11-4b9b-bb74-3c86036c8d20,TCGA-MH-A855,14f96a1d-5a17-44b1-8c8a-bfabb1413432,TCGA-MH-A855-01A,Primary Tumor,TCGA-KIRP,KIRP,Kidney renal papillary cell carcinoma
25e923e1-24ce-44a9-934c-20df73d33686,acfb6bc0-2972-48f8-9926-5a5ef46e8d86.rna_seq.augmented_star_gene_counts.tsv,4229913,87c5e2cb-0b2e-4f40-927b-5e1ad7de4824,TCGA-P4-A5E6,6906dc9f-4bf1-43ac-9967-1b23f3de1196,TCGA-P4-A5E6-01A,Primary Tumor,TCGA-KIRP,KIRP,Kidney renal papillary cell carcinoma


Conteo por tipo de cáncer:


n_archivos,count
BRCA,1231
KIRC,614
LUAD,601
UCEC,589
THCA,572
HNSC,566
LUSC,562
PRAD,554
LGG,534
COAD,524


In [0]:
# Filtrar dataset oficial: 18 clases + Primary Tumor

df_metadatos_oficial = df_metadatos[
    df_metadatos["cancer_type"].isin(CLASES_PRINCIPALES)
].copy()

df_metadatos_oficial = df_metadatos_oficial[
    df_metadatos_oficial["sample_type"]
    .astype(str)
    .str.contains("Primary Tumor", case=False, na=False)
].copy()

print("Archivos oficiales después de filtrar 18 clases + Primary Tumor:")
print(df_metadatos_oficial.shape)

display(
    df_metadatos_oficial["cancer_type"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "cancer_type", "cancer_type": "n_archivos"})
)

Archivos oficiales después de filtrar 18 clases + Primary Tumor:
(8469, 11)


n_archivos,count
BRCA,1111
UCEC,553
KIRC,541
LUAD,540
HNSC,520
LGG,516
LUSC,511
THCA,505
PRAD,501
COAD,481


In [0]:
# Guardar metadatos en zona raw/metadata
metadata_completo_path = f"{RAW_METADATA_PATH}/metadatos_tcga_completo.csv"
metadata_oficial_path = f"{RAW_METADATA_PATH}/metadatos_tcga_oficial_18_clases.csv"

df_metadatos.to_csv(metadata_completo_path, index=False)
df_metadatos_oficial.to_csv(metadata_oficial_path, index=False)

print("Metadatos guardados en:")
print(metadata_completo_path)
print(metadata_oficial_path)

Metadatos guardados en:
/Volumes/workspace/default/tcga_cancer_ml/raw/metadata/metadatos_tcga_completo.csv
/Volumes/workspace/default/tcga_cancer_ml/raw/metadata/metadatos_tcga_oficial_18_clases.csv


In [0]:
# Catalogar metadatos como tablas Delta

spark_metadatos = spark.createDataFrame(df_metadatos)
spark_metadatos_oficial = spark.createDataFrame(df_metadatos_oficial)

(
    spark_metadatos
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.raw_tcga_metadatos_completo")
)

(
    spark_metadatos_oficial
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.raw_tcga_metadatos_oficial_18_clases")
)

In [0]:
# Descargar archivos RNA-Seq desde GDC hacia raw/rnaseq

def descargar_archivo_gdc(file_id, file_name, carpeta_destino, max_reintentos=3):
    """
    Descarga un archivo individual desde GDC usando file_id.
    Guarda cada archivo dentro de una carpeta con su file_id.
    """

    carpeta_file_id = Path(carpeta_destino) / file_id
    carpeta_file_id.mkdir(parents=True, exist_ok=True)

    ruta_salida = carpeta_file_id / file_name

    if ruta_salida.exists() and ruta_salida.stat().st_size > 0:
        return "ya_existe", str(ruta_salida)

    url = f"{DATA_ENDPT}/{file_id}"

    for intento in range(1, max_reintentos + 1):
        try:
            with requests.get(url, stream=True, timeout=180) as r:
                r.raise_for_status()

                with open(ruta_salida, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)

            return "descargado", str(ruta_salida)

        except Exception as e:
            print(f"Error descargando {file_id} intento {intento}: {e}")
            time.sleep(5)

    return "error", str(ruta_salida)


# Para descargar todo, dejar en None.
MAX_DESCARGAS = None

df_descarga = df_metadatos_oficial.copy()

if MAX_DESCARGAS is not None:
    df_descarga = df_descarga.head(MAX_DESCARGAS)

print("Archivos a descargar:", len(df_descarga))

registros_descarga = []

for idx, fila in df_descarga.iterrows():
    file_id = fila["file_id"]
    file_name = fila["file_name"]

    estado, ruta_local = descargar_archivo_gdc(
        file_id=file_id,
        file_name=file_name,
        carpeta_destino=RAW_RNASEQ_PATH
    )

    registros_descarga.append({
        "file_id": file_id,
        "file_name": file_name,
        "estado_descarga": estado,
        "ruta_local": ruta_local
    })

    if len(registros_descarga) % 50 == 0:
        print(f"Procesados {len(registros_descarga)} de {len(df_descarga)}")

df_descargas = pd.DataFrame(registros_descarga)

print("Resumen de descargas:")
display(df_descargas["estado_descarga"].value_counts().reset_index())

display(df_descargas.head())

Archivos a descargar: 8469
Procesados 50 de 8469
Procesados 100 de 8469
Procesados 150 de 8469
Procesados 200 de 8469
Error descargando 986a8890-65eb-4fc6-bd57-c137d63567b4 intento 1: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Procesados 250 de 8469
Procesados 300 de 8469
Procesados 350 de 8469
Procesados 400 de 8469
Procesados 450 de 8469
Procesados 500 de 8469
Procesados 550 de 8469
Procesados 600 de 8469
Procesados 650 de 8469
Procesados 700 de 8469
Procesados 750 de 8469
Procesados 800 de 8469
Procesados 850 de 8469
Procesados 900 de 8469
Procesados 950 de 8469
Procesados 1000 de 8469
Procesados 1050 de 8469
Procesados 1100 de 8469
Procesados 1150 de 8469
Procesados 1200 de 8469
Procesados 1250 de 8469
Procesados 1300 de 8469
Procesados 1350 de 8469
Procesados 1400 de 8469
Procesados 1450 de 8469
Procesados 1500 de 8469
Procesados 1550 de 8469
Procesados 1600 de 8469
Procesados 1650 de 8469
Procesados 1700 de 8469
Procesados 1750 de

estado_descarga,count
descargado,8468
ya_existe,1


file_id,file_name,estado_descarga,ruta_local
744a6d3d-b666-49aa-8d26-47f34e3d1eb5,94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.augmented_star_gene_counts.tsv,descargado,/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/744a6d3d-b666-49aa-8d26-47f34e3d1eb5/94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.augmented_star_gene_counts.tsv
4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5,df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.augmented_star_gene_counts.tsv,descargado,/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5/df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.augmented_star_gene_counts.tsv
1ace2a0c-773d-45b5-8fd6-968c88731bbb,c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.augmented_star_gene_counts.tsv,descargado,/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/1ace2a0c-773d-45b5-8fd6-968c88731bbb/c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.augmented_star_gene_counts.tsv
7db46624-6d07-4f49-b0ff-18bb41375de9,b1f7af2a-a39b-44c7-a077-e1df6b18268a.rna_seq.augmented_star_gene_counts.tsv,descargado,/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/7db46624-6d07-4f49-b0ff-18bb41375de9/b1f7af2a-a39b-44c7-a077-e1df6b18268a.rna_seq.augmented_star_gene_counts.tsv
25e923e1-24ce-44a9-934c-20df73d33686,acfb6bc0-2972-48f8-9926-5a5ef46e8d86.rna_seq.augmented_star_gene_counts.tsv,descargado,/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/25e923e1-24ce-44a9-934c-20df73d33686/acfb6bc0-2972-48f8-9926-5a5ef46e8d86.rna_seq.augmented_star_gene_counts.tsv


In [0]:
# Guardar manifiesto de descargas

manifest_descargas_path = f"{RAW_METADATA_PATH}/manifest_descargas_gdc.csv"

df_descargas.to_csv(manifest_descargas_path, index=False)

spark_descargas = spark.createDataFrame(df_descargas)

(
    spark_descargas
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.raw_tcga_manifest_descargas")
)

print("Manifiesto de descargas guardado en:")
print(manifest_descargas_path)

print("Tabla Delta creada:")
print("workspace.default.raw_tcga_manifest_descargas")

Manifiesto de descargas guardado en:
/Volumes/workspace/default/tcga_cancer_ml/raw/metadata/manifest_descargas_gdc.csv
Tabla Delta creada:
workspace.default.raw_tcga_manifest_descargas


In [0]:
# Validar archivos descargados en raw/rnaseq
archivos_descargados = []

for root, dirs, files in os.walk(RAW_RNASEQ_PATH):
    for file in files:
        if file.endswith(".tsv") and "rna_seq.augmented_star_gene_counts" in file.lower():
            archivos_descargados.append(os.path.join(root, file))

print("Archivos RNA-Seq descargados encontrados:", len(archivos_descargados))

print("Primeros archivos:")
for archivo in archivos_descargados[:10]:
    print(archivo)

Archivos RNA-Seq descargados encontrados: 13742
Primeros archivos:
/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/1ace2a0c-773d-45b5-8fd6-968c88731bbb/c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.augmented_star_gene_counts.tsv
/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/1ace2a0c-773d-45b5-8fd6-968c88731bbb/c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.augmented_star_gene_counts.tsv
/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/1ace2a0c-773d-45b5-8fd6-968c88731bbb/c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.augmented_star_gene_counts.tsv
/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/03443702-1483-4701-89c9-265a55a0ee68/4b5cf3f4-cd52-4a60-8b52-053922decf1b.rna_seq.augmented_star_gene_counts.tsv
/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/03443702-1483-4701-89c9-265a55a0ee68/4b5cf3f4-cd52-4a60-8b52-053922decf1b.rna_seq.augmented_star_gene_counts.tsv
/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/03443702-1483-4701-89c9-265a55a0ee68/4b5cf3f4-cd52